# Step 2: Build the dataset

This notebook takes the raw files from Step 1 and makes one clean CSV with **one row per game**.

1. Load the schedule and the raw odds
2. Remove bad lines (lines that are way off from the other sportsbooks)
3. Match each SBR game to the MLB `game_pk`
4. Reshape so each game is one row, with separate columns for each sportsbook
5. Add implied probabilities for the moneyline
6. Run a few checks and save

**Note on "close":** SBR shows an opening line and a "current" line. For past games the current line is
the last line SBR has saved, which is usually the closing line, but SBR doesn't guarantee it's the exact
final price before first pitch. We call it `close` to keep the column names short.

In [51]:
import glob
import os
import pandas as pd
from odds_helpers import BOOKS, fix_team_name, implied_probability


## 1. Load the data

In [52]:
schedule = pd.read_csv("data/schedule.csv")

frames = []
scraped_dates = []
for file in sorted(glob.glob("data/raw_odds/*.csv")):
    scraped_dates.append(os.path.basename(file).replace(".csv", ""))
    try:
        frames.append(pd.read_csv(file))
    except pd.errors.EmptyDataError:
        print("No odds in", file)
odds = pd.concat(frames, ignore_index=True)

# Drop exact duplicate rows
odds = odds.drop_duplicates()

# SBR uses 0 when a book has no line, so treat 0 as missing.
# Odds bigger than +/-1000 are live in-game lines (pregame MLB odds are almost never that big), so remove those too.
odds_columns = ["open_away_odds", "open_home_odds", "open_over_odds", "open_under_odds",
                "close_away_odds", "close_home_odds", "close_over_odds", "close_under_odds"]
for col in odds_columns:
    odds.loc[odds[col] == 0, col] = None
    odds.loc[odds[col].abs() > 1000, col] = None

# Run lines should be +1.5 or -1.5, and totals should be between 5 and 15 runs.
# Anything else is a live in-game line or junk (like 999), so blank out that line and its odds.
for when in ["open", "close"]:
    bad_spread = odds[f"{when}_home_spread"].notna() & (odds[f"{when}_home_spread"].abs() != 1.5)
    odds.loc[bad_spread, [f"{when}_home_spread", f"{when}_away_odds", f"{when}_home_odds"]] = None

    bad_total = (odds[f"{when}_total"] < 5) | (odds[f"{when}_total"] > 15)
    odds.loc[bad_total, [f"{when}_total", f"{when}_over_odds", f"{when}_under_odds"]] = None

    # Pregame run line odds stay within about +/-300 and total odds within about +/-150.
    # Anything past +/-350 (run line) or +/-200 (total) is a live in-game line, so blank out that line and its odds.
    lopsided_rl = (odds["market"] == "runline") & ((odds[f"{when}_away_odds"].abs() > 350) | (odds[f"{when}_home_odds"].abs() > 350))
    odds.loc[lopsided_rl, [f"{when}_home_spread", f"{when}_away_odds", f"{when}_home_odds"]] = None

    lopsided_total = (odds[f"{when}_over_odds"].abs() > 200) | (odds[f"{when}_under_odds"].abs() > 200)
    odds.loc[lopsided_total, [f"{when}_total", f"{when}_over_odds", f"{when}_under_odds"]] = None

# If one side of a line was removed, the other side came from the same bad line, so remove it too
# (along with the run line or total itself)
for when in ["open", "close"]:
    pairs = [[f"{when}_away_odds", f"{when}_home_odds", f"{when}_home_spread"],
             [f"{when}_over_odds", f"{when}_under_odds", f"{when}_total"]]
    for side1, side2, line in pairs:
        one_missing = odds[side1].isna() | odds[side2].isna()
        odds.loc[one_missing, [side1, side2, line]] = None

        # A real pregame line always has a margin: both sides add up to a bit more than 100%.
        # Under 100% or over 112% means a live in-game line, so remove it.
        margin = odds[side1].apply(implied_probability).astype(float) + odds[side2].apply(implied_probability).astype(float)
        odds.loc[(margin < 1) | (margin > 1.12), [side1, side2, line]] = None

# If not every date has been scraped yet, only use the dates we have.
# Dates with an empty file (SBR had no odds) are kept, so those games stay in the dataset with blank odds.
schedule = schedule[schedule["date"].isin(scraped_dates)]

print(len(odds), "odds rows for", len(schedule), "games")
odds.head()

No odds in data/raw_odds/2022-10-23.csv
No odds in data/raw_odds/2022-11-05.csv
No odds in data/raw_odds/2023-10-04.csv
237497 odds rows for 12359 games


,date,market,sbr_game_id,start_time,away,home,book,open_away_odds,open_home_odds,open_home_spread,open_total,open_over_odds,open_under_odds,close_away_odds,close_home_odds,close_home_spread,close_total,close_over_odds,close_under_odds
0,2022-04-07,moneyline,264292,2022-04-07T18:20:00+00:00,Milwaukee Brewers,Chicago Cubs,betmgm,-150.0,125.0,NaN,NaN,NaN,NaN,-185.0,150.0,NaN,NaN,NaN,NaN
1,2022-04-07,moneyline,264292,2022-04-07T18:20:00+00:00,Milwaukee Brewers,Chicago Cubs,bet365,-160.0,140.0,NaN,NaN,NaN,NaN,-180.0,160.0,NaN,NaN,NaN,NaN
2,2022-04-07,moneyline,264292,2022-04-07T18:20:00+00:00,Milwaukee Brewers,Chicago Cubs,draftkings,-150.0,130.0,NaN,NaN,NaN,NaN,-180.0,155.0,NaN,NaN,NaN,NaN
3,2022-04-07,moneyline,264292,2022-04-07T18:20:00+00:00,Milwaukee Brewers,Chicago Cubs,fanduel,-154.0,130.0,NaN,NaN,NaN,NaN,-190.0,160.0,NaN,NaN,NaN,NaN
4,2022-04-07,moneyline,264292,2022-04-07T18:20:00+00:00,Milwaukee Brewers,Chicago Cubs,bet_rivers_co,-148.0,125.0,NaN,NaN,NaN,NaN,-177.0,148.0,NaN,NaN,NaN,NaN


## 2. Remove bad lines

Sometimes SBR saves a **live in-game line** as the "current" line, or mixes up the lines for the two games
of a doubleheader. To catch these, we compare each sportsbook's moneyline to the **median of all the
sportsbooks** on SBR for the same game (not just the ones we keep). Books usually agree within 2–3
percentage points, so if a book is more than 8 points off, we blank out that book's line for that game
(moneyline, run line and total), since the run line and total are probably bad too.

Opening and closing lines are checked separately.

In [53]:
ml = odds[odds["market"] == "moneyline"].copy()

for when in ["open", "close"]:
    # No-vig probability of the home team winning, for every book
    away_prob = ml[f"{when}_away_odds"].apply(implied_probability).astype(float)
    home_prob = ml[f"{when}_home_odds"].apply(implied_probability).astype(float)
    ml[f"home_novig_{when}"] = home_prob / (away_prob + home_prob)

    # Median of all the books for the same game
    median = ml.groupby("sbr_game_id")[f"home_novig_{when}"].transform("median")
    ml[f"bad_{when}"] = (ml[f"home_novig_{when}"] - median).abs() > 0.08

# Add the bad_open / bad_close flags to every market for that game and book
odds = odds.merge(ml[["sbr_game_id", "book", "bad_open", "bad_close"]], on=["sbr_game_id", "book"], how="left")

open_columns = [col for col in odds.columns if col.startswith("open_")]
close_columns = [col for col in odds.columns if col.startswith("close_")]
odds.loc[odds["bad_open"] == True, open_columns] = None
odds.loc[odds["bad_close"] == True, close_columns] = None

# Now only keep our sportsbooks
odds = odds[odds["book"].isin(BOOKS.keys())]

print("Games where we blanked out a book's lines:")
ml[ml["book"].isin(BOOKS.keys())].groupby("book")[["bad_open", "bad_close"]].sum()

Games where we blanked out a book's lines:


,bad_open,bad_close
book,,
bet365,6,145
draftkings,35,91
fanduel,15,92


## 3. Match SBR games to MLB games

We match on date, away team and home team. For **doubleheaders** (two games between the same teams on the
same day) this matches each SBR game to both MLB games, so we keep the one with the closest start time.

In [54]:
odds["away"] = odds["away"].apply(fix_team_name)
odds["home"] = odds["home"].apply(fix_team_name)

matched = odds.merge(schedule[["game_pk", "date", "away", "home", "start_time"]],
                     on=["date", "away", "home"], how="left", suffixes=("_sbr", "_mlb"))

# How far apart the SBR and MLB start times are, in minutes
sbr_time = pd.to_datetime(matched["start_time_sbr"], utc=True)
mlb_time = pd.to_datetime(matched["start_time_mlb"], utc=True)
matched["minutes_apart"] = (sbr_time - mlb_time).abs().dt.total_seconds() / 60

# SBR games that aren't in the schedule don't match anything: spring training games SBR lists on the days
# of the 2024 Seoul and 2025 Tokyo openers, and games that were postponed
not_matched = matched[matched["game_pk"].isna()]
print(not_matched["sbr_game_id"].nunique(), "SBR games didn't match an MLB game")
matched = matched[matched["game_pk"].notna()].copy()
matched["game_pk"] = matched["game_pk"].astype(int)

# Keep the closest match for each SBR game, and the closest SBR game for each MLB game
matched = matched.sort_values("minutes_apart")
matched = matched.drop_duplicates(subset=["sbr_game_id", "book", "market"], keep="first")
matched = matched.drop_duplicates(subset=["game_pk", "book", "market"], keep="first")

38 SBR games didn't match an MLB game


American odds are always whole numbers, but pandas shows them as decimals (like `-135.0`) when some
values are missing. `Int64` (capital I) is the pandas integer type that allows missing values.

In [55]:
for col in odds_columns:
    matched[col] = matched[col].round().astype("Int64")

## 4. Reshape to one row per game

Column names follow the pattern `book_market_side_open/close`. Books: `b365` = bet365, `dk` = DraftKings, `fd` = FanDuel.
Markets: `ml` = moneyline, `rl` = run line, `total` = over/under.
For the run line we only keep the home spread, since the away spread is always the opposite.

In [56]:
game_type_names = {"R": "regular", "F": "wild card", "D": "division series",
                   "L": "league championship", "W": "world series"}

games = schedule[["game_pk", "date", "season", "game_type", "away", "home", "start_time"]].copy()
games["game_type"] = games["game_type"].map(game_type_names)
games = games.rename(columns={"start_time": "start_time_utc"})

for book, short in BOOKS.items():
    book_odds = matched[matched["book"] == book]

    # Moneyline
    ml = book_odds[book_odds["market"] == "moneyline"]
    ml = ml[["game_pk", "open_away_odds", "open_home_odds", "close_away_odds", "close_home_odds"]]
    ml.columns = ["game_pk", f"{short}_ml_away_open", f"{short}_ml_home_open",
                  f"{short}_ml_away_close", f"{short}_ml_home_close"]
    games = games.merge(ml, on="game_pk", how="left")

    # Run line
    rl = book_odds[book_odds["market"] == "runline"]
    rl = rl[["game_pk", "open_home_spread", "open_away_odds", "open_home_odds",
             "close_home_spread", "close_away_odds", "close_home_odds"]]
    rl.columns = ["game_pk", f"{short}_rl_home_spread_open", f"{short}_rl_away_open", f"{short}_rl_home_open",
                  f"{short}_rl_home_spread_close", f"{short}_rl_away_close", f"{short}_rl_home_close"]
    games = games.merge(rl, on="game_pk", how="left")

    # Total
    total = book_odds[book_odds["market"] == "total"]
    total = total[["game_pk", "open_total", "open_over_odds", "open_under_odds",
                   "close_total", "close_over_odds", "close_under_odds"]]
    total.columns = ["game_pk", f"{short}_total_open", f"{short}_total_over_open", f"{short}_total_under_open",
                     f"{short}_total_close", f"{short}_total_over_close", f"{short}_total_under_close"]
    games = games.merge(total, on="game_pk", how="left")

games = games.sort_values(["date", "start_time_utc"]).reset_index(drop=True)
games.head()

,game_pk,date,season,game_type,away,home,start_time_utc,b365_ml_away_open,b365_ml_home_open,b365_ml_away_close,...,fd_rl_home_open,fd_rl_home_spread_close,fd_rl_away_close,fd_rl_home_close,fd_total_open,fd_total_over_open,fd_total_under_open,fd_total_close,fd_total_over_close,fd_total_under_close
0,663178,2022-04-07,2022,regular,Milwaukee Brewers,Chicago Cubs,2022-04-07T18:20:00Z,-160,140,-180,...,-140,1.5,-110,-110,7.0,-110,-110,9.0,-108,-112
1,662766,2022-04-07,2022,regular,Cleveland Guardians,Kansas City Royals,2022-04-07T20:10:00Z,-125,105,-125,...,-194,1.5,140,-170,8.0,-114,-106,7.5,100,-122
2,662021,2022-04-07,2022,regular,Pittsburgh Pirates,St. Louis Cardinals,2022-04-07T20:15:00Z,180,-220,135,...,114,-1.5,-154,128,7.5,-105,-115,8.0,-102,-120
3,662571,2022-04-07,2022,regular,New York Mets,Washington Nationals,2022-04-07T23:05:00Z,-135,115,<NA>,...,-160,1.5,126,-152,8.5,-120,-102,9.0,-122,100
4,661577,2022-04-07,2022,regular,Cincinnati Reds,Atlanta Braves,2022-04-08T00:08:00Z,175,-210,165,...,110,-1.5,-140,110,7.5,-114,-106,8.5,-104,-122


## 5. Implied probabilities (moneyline)

- **prob**: the probability the odds imply. The two sides add up to a bit more than 100%; the extra is the
  sportsbook's cut (the "vig").
- **novig**: the same probabilities scaled so the two sides add up to exactly 100%.

In [57]:
for short in BOOKS.values():
    for when in ["open", "close"]:
        away_prob = games[f"{short}_ml_away_{when}"].apply(implied_probability).astype(float)
        home_prob = games[f"{short}_ml_home_{when}"].apply(implied_probability).astype(float)

        games[f"{short}_ml_away_prob_{when}"] = away_prob.round(4)
        games[f"{short}_ml_home_prob_{when}"] = home_prob.round(4)
        games[f"{short}_ml_away_novig_{when}"] = (away_prob / (away_prob + home_prob)).round(4)
        games[f"{short}_ml_home_novig_{when}"] = (home_prob / (away_prob + home_prob)).round(4)

games[["away", "home", "dk_ml_away_close", "dk_ml_home_close",
       "dk_ml_away_prob_close", "dk_ml_home_prob_close",
       "dk_ml_away_novig_close", "dk_ml_home_novig_close"]].head()

,away,home,dk_ml_away_close,dk_ml_home_close,dk_ml_away_prob_close,dk_ml_home_prob_close,dk_ml_away_novig_close,dk_ml_home_novig_close
0,Milwaukee Brewers,Chicago Cubs,-180,155,0.6429,0.3922,0.6211,0.3789
1,Cleveland Guardians,Kansas City Royals,-130,110,0.5652,0.4762,0.5427,0.4573
2,Pittsburgh Pirates,St. Louis Cardinals,170,-200,0.3704,0.6667,0.3571,0.6429
3,New York Mets,Washington Nationals,-135,115,0.5745,0.4651,0.5526,0.4474
4,Cincinnati Reds,Atlanta Braves,170,-200,0.3704,0.6667,0.3571,0.6429


## 6. Checks

In [58]:
# Every game should only be in the dataset once
assert games["game_pk"].is_unique

# Every played game from the schedule should be in the dataset (with or without odds)
assert len(games) == len(schedule)

# No-vig probabilities should add up to 1
for short in BOOKS.values():
    total_prob = (games[f"{short}_ml_away_novig_close"] + games[f"{short}_ml_home_novig_close"]).dropna()
    assert ((total_prob - 1).abs() < 0.001).all()

# American odds should never be between -100 and +100, or bigger than +/-1000
for col in odds_columns:
    assert not matched[col].between(-99, 99).any()
    assert not (matched[col].abs() > 1000).any()

# Run lines should all be +/-1.5, and totals between 5 and 15
for when in ["open", "close"]:
    assert matched[f"{when}_home_spread"].dropna().abs().eq(1.5).all()
    assert matched[f"{when}_total"].dropna().between(5, 15).all()

    # Run line odds should be within +/-350, and total odds within +/-200
    assert not (matched.loc[matched["market"] == "runline", [f"{when}_away_odds", f"{when}_home_odds"]].abs() > 350).any().any()
    assert not (matched[[f"{when}_over_odds", f"{when}_under_odds"]].abs() > 200).any().any()

# Both sides of every line should be there, or both blank
for short in BOOKS.values():
    for when in ["open", "close"]:
        for side1, side2 in [("ml_away", "ml_home"), ("rl_away", "rl_home"), ("total_over", "total_under")]:
            assert (games[f"{short}_{side1}_{when}"].isna() == games[f"{short}_{side2}_{when}"].isna()).all()

print("Checks passed")

Checks passed


How many games have closing moneyline odds from each sportsbook (% of games per season):

In [59]:
close_columns = [f"{short}_ml_home_close" for short in BOOKS.values()]
has_odds = games[close_columns].notna()
has_odds["season"] = games["season"]
(has_odds.groupby("season").mean() * 100).round(1)

,b365_ml_home_close,dk_ml_home_close,fd_ml_home_close
season,,,
2022,94.3,96.9,96.3
2023,99.1,98.3,98.3
2024,97.7,97.8,98.2
2025,98.7,97.7,97.9
2026,99.1,99.8,99.9


## 7. Save

In [60]:
games.to_csv("data/mlb_odds_2022_2026.csv", index=False)
print(f"Saved {len(games)} games and {len(games.columns)} columns to data/mlb_odds_2022_2026.csv")

Saved 12359 games and 79 columns to data/mlb_odds_2022_2026.csv
